In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor, XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    roc_auc_score,
)

## Load data 

In [2]:
df = pd.read_csv("data/cibil_score/cibil_score.csv")
df = df.drop(columns=["Unnamed: 0"])

# normalize column names
df.columns = [col.lower().strip() for col in df.columns]

print(df.shape)
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
df.head()

(51336, 87)
Duplicate rows: 0


,prospectid,total_tl,tot_closed_tl,tot_active_tl,total_tl_opened_l6m,tot_tl_closed_l6m,pct_tl_open_l6m,pct_tl_closed_l6m,pct_active_tl,pct_closed_tl,...,pct_cc_enq_l6m_of_l12m,pct_pl_enq_l6m_of_ever,pct_cc_enq_l6m_of_ever,max_unsec_exposure_inpct,hl_flag,gl_flag,last_prod_enq2,first_prod_enq2,credit_score,approved_flag
0,1,5,4,1,0,0,0.000,0.0,0.200,0.800,...,0.0,0.0,0.0,13.333,1,0,PL,PL,696,P2
1,2,1,0,1,0,0,0.000,0.0,1.000,0.000,...,0.0,0.0,0.0,0.860,0,0,ConsumerLoan,ConsumerLoan,685,P2
2,3,8,0,8,1,0,0.125,0.0,1.000,0.000,...,0.0,0.0,0.0,5741.667,1,0,ConsumerLoan,others,693,P2
3,4,1,0,1,1,0,1.000,0.0,1.000,0.000,...,0.0,0.0,0.0,9.900,0,0,others,others,673,P2
4,5,3,2,1,0,0,0.000,0.0,0.333,0.667,...,0.0,0.0,0.0,-99999.000,0,0,AL,AL,753,P1


## Evaluation Functions

In [3]:
def evaluate_regression(model, X_te, y_te):

    y_pred = model.predict(X_te)

    return {
        "r2": r2_score(y_te, y_pred),
        "mae": mean_absolute_error(y_te, y_pred),
        "rmse": root_mean_squared_error(y_te, y_pred),
    }


def evaluate_classification(model, X_te, y_te, binary):

    y_pred = model.predict(X_te)

    metrics = {
        "accuracy": accuracy_score(y_te, y_pred),

        "f1_macro": f1_score(
            y_te,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "precision_macro": precision_score(
            y_te,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "recall_macro": recall_score(
            y_te,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "precision_weighted": precision_score(
            y_te,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "recall_weighted": recall_score(
            y_te,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "f1_weighted": f1_score(
            y_te,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "confusion_matrix": confusion_matrix(
            y_te,
            y_pred
        )
    }

    if binary and hasattr(model, "predict_proba"):

        y_prob = model.predict_proba(X_te)[:, 1]

        metrics["roc_auc"] = roc_auc_score(
            y_te,
            y_prob
        )

    return metrics


## Remove the targets

In [4]:
X = df.drop(columns=["approved_flag", "credit_score"])

y_linear = df["credit_score"].astype(float)

y_multiclass = df["approved_flag"].astype(str)

binary_map = {
    "P1": 1,
    "P2": 1,
    "P3": 0,
    "P4": 0
}

y_binary = df["approved_flag"].map(binary_map)

assert y_binary.isna().sum() == 0, \
    "approved_flag has values outside P1-P4"

print(y_multiclass.value_counts())
print(y_binary.value_counts())

approved_flag
P2    32199
P3     7452
P4     5882
P1     5803
Name: count, dtype: int64
approved_flag
1    38002
0    13334
Name: count, dtype: int64


## Train/test split

In [5]:
# Regression
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X,
    y_linear,
    test_size=0.2,
    random_state=42
)

# Binary classification
X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X,
    y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

# Multiclass classification
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X,
    y_multiclass,
    test_size=0.2,
    random_state=42,
    stratify=y_multiclass
)

## Identify numerical and categorical columns

In [6]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("\nNumber of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))


Number of numerical features: 80
Number of categorical features: 5


## Preprocessor

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)



## XGBoost Regressor

In [10]:
xgb_reg = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        ("model", XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

xgb_reg.fit(
    X_train_reg,
    y_train_reg,
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](85,)","['prospectid','total_tl','tot_closed_tl',...,'gl_flag','last_prod_enq2', 'first_prod_enq2']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,85
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of c

In [11]:
reg_results = evaluate_regression(
    xgb_reg,
    X_test_reg,
    y_test_reg
)

print(reg_results)

{'r2': 0.9032289388224607, 'mae': 5.451751720872797, 'rmse': 6.362284483708275}


## Binary classification using XGBClassifier

## Multiclass classification using XGBClassifier

**Regularization**

XGBoost builds trees sequentially in a chain (boosting), where each new tree tries to fix the mistakes of the previous one.
Because of this sequential nature, XGBoost can easily overfit if left unchecked, so it includes mathematically advanced **L1** and **L2** regularization penalties directly inside its loss function


**1. Mathematical Regularization (XGBoost Only)**

XGBoost alters the math of how it evaluates a tree split by adding penalties directly to its loss function. Random Forest does not have these.

**reg_lambda (L2 Regularization):** Adds a Ridge-like penalty to the leaf weights. Increasing this shrinks the prediction values of individual leaves toward zero, making the model more conservative.

**reg_alpha (L1 Regularization):** Adds a Lasso-like penalty to the leaf weights. If a feature or a split provides very little benefit, this will force its leaf weight exactly to zero, effectively acting as an automated feature selector.

**2. Structural Regularization (Both, but different names)**

Both algorithms can control the physical size of the trees, but the parameters differ:

| Regularization Goal | Random Forest Parameter | XGBoost Equivalent Parameter | How it works |
|---|---|---|---|
| Limit tree depth | max_depth | max_depth | XGBoost trees are usually much shallower (e.g., depth 3 to 6) than Random Forest trees (depth 15+). |
| Minimum node size	| min_samples_leaf | min_child_weight | In XGBoost, this dictates the minimum sum of instance weights (Hessian) needed in a child node, rather than a raw count of samples. |
| Strict split threshold | None (or ccp_alpha) | gamma (or min_split_loss) | A split is made only if the loss reduction exceeds this value. Higher gamma = aggressive pruning. |

**3. Randomness & Subsampling (Both)**

Both algorithms inject randomness to prevent trees from memorizing the training data:
Feature Subsampling: Random Forest uses max_features. XGBoost uses colsample_bytree or colsample_bylevel to choose what percentage of features each tree or split can see.
Row Subsampling: Random Forest always bootstraps rows by default. XGBoost allows you to tune this manually with subsample (e.g., setting it to 0.8 forces each tree to train on a random 80% subset of rows).

**4. Learning Rate Regularization (XGBoost Only)**

learning_rate (or eta): Because XGBoost adds trees together sequentially, it uses a step-size shrinkage. After every tree is built, its contribution is multiplied by this rate (e.g., 0.05). This slows down the learning process, ensuring that no single tree dominates the model, requiring later trees to fine-tune the predictions.

New Prediction = Old Prediction + η × Tree Output

where η (eta) is usually between 0.01 to 0.3

Smaller learning rates require more trees but often improve generalization.

